In [6]:
import json
import pandas as pd

# ------------------------------------------------------------------
# 1. Example: load the JSON (replace with your actual file path)
# ------------------------------------------------------------------
# If you already have the JSON in a variable, skip this and just set `data` directly.
with open("drug_deepsearch_json_final/Paclitaxel.json", "r") as f:
    data = json.load(f)

# ------------------------------------------------------------------
# 2. Example: create/read the DataFrame
#    (replace this with pd.read_csv(...) or your actual data source)
# ------------------------------------------------------------------

df = pd.read_excel("Deep_search_results/deepsearch_3_ADMINISTRATION_regulation_effect_rationale_citation_tab.xlsx")

# ------------------------------------------------------------------
# 3. Filter for Paclitaxel
# ------------------------------------------------------------------
drug_name = "Paclitaxel"
df_drug = df[df["Drug"] == drug_name]

# ------------------------------------------------------------------
# 4. Build nested structure under pathway_sets_annotations
#    Format:
# "Administration_of_paclitaxel": {
#   "Before": {
#     "Pathway_regulation": {
#       "Up": {
#         "sensitive": [[rationale, citations], ...],
#         "resistance": [[rationale, citations], ...]
#       },
#       "Down": { ... }
#     }
#   },
#   "After": { ... }
# }
# ------------------------------------------------------------------

# Ensure top-level key exists
if "pathway_sets_annotations" not in data:
    data["pathway_sets_annotations"] = {}

for _, row in df_drug.iterrows():
    pathway = row["Pathway"]
    admin = row["Administration"]   # "Before" / "After"
    reg = row["Regulation"]         # "Up" / "Down"
    effect = row["Effect"]          # "Sensitive" / "Resistance"
    rationale = row["Rationale"]

    # Normalize keys to match your schema
    admin_key = admin  # e.g. "Before", "After"
    reg_key = reg      # e.g. "Up", "Down"
    effect_key = effect.lower()  # "sensitive" / "resistance"

    # Make sure the pathway exists in annotations
    if pathway not in data["pathway_sets_annotations"]:
        data["pathway_sets_annotations"][pathway] = {}

    pathway_dict = data["pathway_sets_annotations"][pathway]

    # Key for this drug under this pathway
    admin_of_drug_key = f"Administration_of_{drug_name.lower()}"  # or keep case if you prefer

    # Initialize nested structure with setdefault
    admin_root = pathway_dict.setdefault(admin_of_drug_key, {})
    admin_block = admin_root.setdefault(admin_key, {})
    reg_block = admin_block.setdefault("Pathway_regulation", {})
    reg_dict = reg_block.setdefault(reg_key, {})
    effect_list = reg_dict.setdefault(effect_key, [])

    # Append [rationale, citations] for this row
    # citations can be filled later; here we use an empty list placeholder.
    effect_list.append([rationale, []])

# ------------------------------------------------------------------
# 5. Save or inspect the updated JSON
# ------------------------------------------------------------------
with open("drug_json_final/Paclitaxel_final.json", "w") as f:
    json.dump(data, f, indent=2)

# For quick inspection:
# print(json.dumps(data, indent=2))


In [7]:
import json
import pandas as pd

# -----------------------------
# 1. Load your JSON
# -----------------------------
# If your JSON is in a file, e.g. "pathway_annotations.json"
with open("drug_deepsearch_json_final/Paclitaxel.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# data is expected to look like:
# {
#   "pathway_sets_annotations": {
#       "HALLMARK_G2M_CHECKPOINT": {
#           "description": "...",
#           "interpretation": "..."
#       },
#       "HALLMARK_MITOTIC_SPINDLE": {
#           "description": "...",
#           "interpretation": "..."
#       }
#   }
# }

# -----------------------------
# 2. Load or define your DataFrame
# -----------------------------
# If your table is in a CSV:
# df = pd.read_csv("paclitaxel_table.csv", sep="\t")  # or appropriate separator

# If you already have df in memory, just make sure columns are:
# ["Drug", "Pathway", "Administration", "Regulation", "Effect", "Rationale", "Ref(s)"]

# For illustration only, here’s a minimal construct from your snippet:
# data_rows = [
#     {
#         "Drug": "Paclitaxel",
#         "Pathway": "HALLMARK_G2M_CHECKPOINT",
#         "Administration": "Before",
#         "Regulation": "Up",
#         "Effect": "Resistance",
#         "Rationale": "High G2/M activity can coexist with robust spindle checkpoints and chromosomal instability programs that let HR+/HER2– and TNBC cells tolerate mitotic errors, reducing paclitaxel lethality.",
#         "Ref(s)": "Swanton et al., Cancer Cell 2007. PMID: 17560332; Swanton 2009. PMID: 19458043; Jordan & Wilson 2004. PMID: 15057285"
#     },
#     {
#         "Drug": "Paclitaxel",
#         "Pathway": "HALLMARK_G2M_CHECKPOINT",
#         "Administration": "Before",
#         "Regulation": "Up",
#         "Effect": "Sensitive",
#         "Rationale": "In many Luminal B and basal-like/TNBC tumors, high G2/M signature reflects many actively dividing cells, increasing the number of mitotic microtubule targets that paclitaxel can kill.",
#         "Ref(s)": "Jordan & Wilson 2004. PMID: 15057285; Nakayama et al. PMID: 19239702"
#     },
#     # ... add the rest of your rows here ...
# ]
df = pd.read_excel("Deep_search_results/deepsearch_3_ADMINISTRATION_regulation_effect_rationale_citation_tab.xlsx")

# -----------------------------
# 3. Filter for Paclitaxel
# -----------------------------
df_pac = df[df["Drug"] == "Paclitaxel"].copy()

# -----------------------------
# 4. Build nested structure under each pathway
#    under the key "Administration_of_paclitaxel"
# -----------------------------
ADMIN_KEY = "Administration_of_paclitaxel"
PATHWAY_ROOT = "pathway_sets_annotations"

# Make sure the top-level key exists
if PATHWAY_ROOT not in data:
    data[PATHWAY_ROOT] = {}

for _, row in df_pac.iterrows():
    pathway = row["Pathway"]
    administration = row["Administration"]  # "Before" or "After"
    regulation = row["Regulation"]         # "Up" or "Down"
    effect = row["Effect"]                 # "Sensitive" or "Resistance"
    rationale = row["Rationale"]
    refs = row["Ref(s)"]

    # Ensure pathway object exists in JSON
    pathway_obj = data[PATHWAY_ROOT].setdefault(pathway, {})

    # Ensure "Administration_of_paclitaxel" object exists
    admin_obj = pathway_obj.setdefault(ADMIN_KEY, {})

    # Ensure "Before"/"After" level
    admin_section = admin_obj.setdefault(administration, {})

    # Ensure "Pathway_regulation" level
    reg_root = admin_section.setdefault("Pathway_regulation", {})

    # Ensure "Up"/"Down" level
    reg_section = reg_root.setdefault(regulation, {})

    # Ensure "Sensitive"/"Resistance" arrays
    # We’ll follow your requested structure:
    # "Up": {
    #   "Sensitive": [ [rationale, citations], ... ],
    #   "Resistance": [ [rationale, citations], ... ]
    # }
    effect_list = reg_section.setdefault(effect, [])

    # Append [rationale, citations]
    effect_list.append([rationale, refs])

# -----------------------------
# 5. Save the updated JSON (optional)
# -----------------------------
with open("drug_json_final/Paclitaxel_final.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
